# einsum-contraction — worked example 3: Batched bilinear form with three operands

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `einsum-contraction`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

A three-operand einsum can contract several indices at once. The bilinear form `xᵀ M y` per batch element contracts both feature indices `i` and `j` while keeping the batch index `b`, collapsing two vectors and a matrix down to one scalar per sample in a single declarative call.

## Worked solution

We compute a per-sample scalar `s[b] = x[b] · M · y[b]` for `x: (B, I)`, `M: (I, J)`, `y: (B, J)`.

1. Classify the indices. `b` is on `x`, on `y`, and on the output — a batch passthrough. `i` is on `x` and `M` only (not on the output) — contracted. `j` is on `M` and `y` only — contracted.
2. The pattern is `'b i, i j, b j -> b'`. Three operands, three comma-separated index groups, and an output of just `b`.
3. einsum evaluates, for each `b`, `sum_i sum_j x[b,i] * M[i,j] * y[b,j]` — the textbook bilinear form, vectorised across the batch.
4. We compare against an explicit per-sample loop `x[k] @ M @ y[k]` to confirm correctness; the result is `(B,)`.

In [ ]:
import torch as t
import einops

t.manual_seed(2)
x = t.randn(4, 3)
M = t.randn(3, 5)
y = t.randn(4, 5)

def bilinear_form(x, M, y):
    return einops.einsum(x, M, y, 'b i, i j, b j -> b')

out = bilinear_form(x, M, y)
print(out.shape)
manual = t.stack([x[k] @ M @ y[k] for k in range(x.shape[0])])
print('matches manual:', bool(t.allclose(out, manual, atol=1e-4)))